In [0]:
spark.sql("use globalretail_silver")
spark.sql("""
          CREATE TABLE IF NOT EXISTS silver_customer (
            customer_id	int,
            name	string,
            email	string,
            country	string,
            customer_type	string,
            registration_date	date,
            age	int,
            gender	string,
            total_purchases	int,
            customer_segment	string,
            days_since_registration	int,
            last_updated TIMESTAMP)
          """)


In [0]:
%sql
SELECT * FROM silver_customer

In [0]:
from datetime import datetime
last_processed_df = spark.sql("select max(last_updated) as last_updated from silver_customer")
last_processed_timestamp = last_processed_df.collect()[0]['last_updated']

if last_processed_timestamp is None:
  last_processed_timestamp = datetime(1900, 1, 1)

  

In [0]:
print(last_processed_timestamp)

In [0]:
#Create a temporary view of incremental bronze data
spark.sql(f"""
CREATE OR REPLACE TEMPORARY VIEW bronze_incremental AS
SELECT * 
FROM globalretail_bronze.bronze_customer c where c.ingestion_timestamp > '{last_processed_timestamp}'
""")

In [0]:
spark.sql("SELECT * FROM bronze_incremental").show()

In [0]:
%sql
SELECT * FROM bronze_incremental

In [0]:
#Validate email addresses (null or now null)
#Valid age between 18 and 100
#Create customer_segment based on total_purchases >10000 is High Value >5000 is Medium Value Else Low Value
#days since registration
#days since last purchase
#total_purchase should not be negative number

In [0]:
spark.sql(f"""
CREATE OR REPLACE TEMPORARY VIEW silver_incremental AS
SELECT
  customer_id,
  name,
  email,
  country,
  customer_type,
  registration_date,
  age,
  gender,
  total_purchases,
  CASE 
    WHEN total_purchases > 10000 THEN 'High Value'
    WHEN total_purchases > 5000 THEN 'Medium Value'
    ELSE 'Low Value'
  END AS customer_segment,
  datediff(current_date(), registration_date) AS days_since_registration,
  current_timestamp() AS last_updated
FROM bronze_incremental
WHERE
  email IS NOT NULL
  AND age BETWEEN 18 AND 100
  AND total_purchases >= 0
""")

In [0]:
display(spark.sql("SELECT * FROM silver_incremental"))

In [0]:
spark.sql("""
MERGE INTO silver_customer t
USING silver_incremental s
ON t.customer_id = s.customer_id
WHEN MATCHED THEN
  UPDATE SET *
WHEN NOT MATCHED THEN
  INSERT *
""")

In [0]:
display(spark.sql("SELECT * FROM silver_customer"))